This notebook filters Yellow Taxi and FHVHV to the zones obtained from the previous notebook and filters to the date range too (2024 only)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd

spark = SparkSession.builder \
    .appName("Taxi_Filtering") \
    .master("local[2]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/23 19:43:41 WARN Utils: Your hostname, LAPTOP-V00NHVKA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/23 19:43:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/23 19:43:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reads the theater zones and finds all distinct values.

In [2]:
theaters_zones = pd.read_csv(
    "../data/broadway_data/theaters_zones.csv"
)

theater_zone_ids = (
    theaters_zones["LocationID"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

theater_zone_ids

[230, 48, 100, 142]

Read the yellow taxi data and prints the schema.

In [3]:
yellow = spark.read.parquet("../data/yellow_taxi_data")
yellow.printSchema()


root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



Count the Yellow Taxi data rows before filtering.

In [4]:
yellow_before = yellow.count()

print("Yellow Taxi rows before filtering:", yellow_before)

Yellow Taxi rows before filtering: 41169720


Filter the Yellow Taxi data to only match the zones with theaters in them and a data range in 2024.

In [5]:
yellow_filtered = yellow.filter(
    (
        F.col("PULocationID").isin(theater_zone_ids)
        |
        F.col("DOLocationID").isin(theater_zone_ids)
    )
    &
    (
        F.col("tpep_pickup_datetime") >= "2024-01-01"
    )
    &
    (
        F.col("tpep_pickup_datetime") < "2025-01-01"
    )
)

Prints the row count before and after filtering, also prints the percentage of rows left after filtering.

In [6]:
yellow_after = yellow_filtered.count()

print("Yellow Taxi rows before filtering:", yellow_before)
print("Yellow Taxi rows after filtering:", yellow_after)
print("Percentage retained:",round(yellow_after / yellow_before * 100, 2),"%")

Yellow Taxi rows before filtering: 41169720
Yellow Taxi rows after filtering: 7742679
Percentage retained: 18.81 %


FHVHV: Do the exact same thing but with the FHVHV data.

In [7]:
fhvhv_path = "../data/fhvhv_data"

fhvhv = spark.read.parquet(fhvhv_path)

fhvhv.printSchema()
fhvhv.select(
    F.min("pickup_datetime").alias("min_pickup"),
    F.max("pickup_datetime").alias("max_pickup")
).show()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_

+-------------------+-------------------+
|         min_pickup|         max_pickup|
+-------------------+-------------------+
|2024-01-01 00:00:00|2024-12-31 23:59:59|
+-------------------+-------------------+



In [8]:
fhvhv_before = fhvhv.count()

print("FHVHV rows before filtering:", fhvhv_before)

FHVHV rows before filtering: 239470448


In [9]:
fhvhv_filtered = fhvhv.filter(
    (
        F.col("PULocationID").isin(theater_zone_ids)
    )
    &
    (
        F.col("pickup_datetime") >= "2024-01-01"
    )
    &
    (
        F.col("pickup_datetime") < "2025-01-01"
    )
)

In [10]:
fhvhv_after = fhvhv_filtered.count()

print("FHVHV rows before filtering:", fhvhv_before)
print("FHVHV rows after filtering:", fhvhv_after)
print("Percentage retained:",round(fhvhv_after / fhvhv_before * 100, 2),"%")


FHVHV rows before filtering: 239470448
FHVHV rows after filtering: 8351440
Percentage retained: 3.49 %


Control zone. To later check whether any relationship between Theater District taxi demand and Broadway attendance is theater-specific or just reflects general Midtown/holiday seasonality, filter Yellow Taxi to a comparable Manhattan zone that does not contain a Broadway theater. 

In [11]:
zone_lookup = pd.read_csv("../data/taxi_zones/taxi+_zone_lookup.csv")

control_zone_name = "Murray Hill"  # a zone not in theater_zone_ids

control_zone_id = int(
    zone_lookup.loc[zone_lookup["Zone"] == control_zone_name, "LocationID"].iloc[0]
)
assert control_zone_id not in theater_zone_ids, "Control zone overlaps theater zones"
print(f"Control zone: {control_zone_name} (LocationID {control_zone_id})")

Control zone: Murray Hill (LocationID 170)


In [12]:
control_filtered = yellow.filter(
    (F.col("PULocationID") == control_zone_id)
    &
    (F.col("tpep_pickup_datetime") >= "2024-01-01")
    &
    (F.col("tpep_pickup_datetime") < "2025-01-01")
)

control_after = control_filtered.count()
print("Control zone rows before filtering:", yellow_before)
print("Control zone rows after filtering:", control_after)
print("Percentage retained:", round(control_after / yellow_before * 100, 2), "%")

Control zone rows before filtering: 41169720
Control zone rows after filtering: 1176580
Percentage retained: 2.86 %


In [13]:
control_filtered.write.mode("overwrite").parquet("../data/curated/control_zone")

Save the filtered data for future use.

In [14]:
yellow_filtered.write.mode("overwrite").parquet("../data/curated/yellow_taxi")
fhvhv_filtered.write.mode("overwrite").parquet("../data/curated/fhvhv")